# Szybka Transformata Fouriera (FFT) - Algorytm Cooleya-Tukeya

Autorzy:

Aron Lampart i Witold Klepek

## 1. Wprowadzenie
Algorytm Cooleya-Tukeya to zdecydowanie najpopularniejszy algorytm do wyznaczania szybkiej transformaty Fouriera (FFT). Pozwala on na drastyczne zmniejszenie złożoności obliczeniowej dyskretnej transformaty Fouriera (DFT) z poziomu $O(N^2)$ do $O(N \log N)$.

Opiera się on na metodzie "dziel i zwyciężaj", rekurencyjnie dzieląc transformatę o rozmiarze $N$ na dwie mniejsze transformaty o rozmiarze $N/2$ (złożone z próbek parzystych i nieparzystych).

## 2. Podstawy matematyczne

### Dyskretna Transformata Fouriera (DFT)
Dla dyskretnego sygnału wejściowego $x$ o długości $N$, transformatę definiuje się wzorem:
$$X_k = \sum_{n=0}^{N-1} x_n e^{-j 2 \pi \frac{k n}{N}}$$

### Rozkład Radix-2 Decimation-in-Time (DIT)
W algorytmie Cooleya-Tukeya (w wariancie DIT) sumę tę rozbijamy na operacje dla indeksów parzystych i nieparzystych:
$$X_k = \sum_{m=0}^{\frac{N}{2}-1} x_{2m} e^{-j 2 \pi \frac{k m}{N/2}} + e^{-j \frac{2 \pi}{N} k} \sum_{m=0}^{\frac{N}{2}-1} x_{2m+1} e^{-j 2 \pi \frac{k m}{N/2}}$$

Powyższe równanie zazwyczaj zapisuje się w uproszczeniu za pomocą tzw. czynnika obrotu (*twiddle factor*):
$$X_k = E_k + e^{-j \frac{2 \pi}{N} k} O_k$$
gdzie $E_k$ to wynik dla części parzystej, a $O_k$ dla części nieparzystej.

## 3. Implementacja Algorytmu

Poniższa implementacja w języku Python korzysta z rekurencyjnego podejścia. Obejmuje ona warunek zatrzymania, podział na próbki parzyste/nieparzyste (*split*) oraz pętlę łączącą wyniki za pomocą operacji z tzw. strukturą motylkową (*butterfly*).

In [2]:
import cmath
import math

def fft_cooley_tukey(x):

    N = len(x)

    # 1. Zatrzymanie rekurencji
    if N <= 1:
        return x

    # 2. Split
    even = fft_cooley_tukey(x[0::2])
    odd =  fft_cooley_tukey(x[1::2])

    result = [0] * N

    # 3. Butterfly & Twiddle Factors
    half_N = N // 2
    for k in range(half_N):
        twiddle_factor = cmath.exp(-2j * cmath.pi * k / N)

        step = twiddle_factor * odd[k]

        result[k]          = even[k] + step
        result[k + half_N] = even[k] - step
    return result

if __name__ == "__main__":
    signal = [1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0]

    print(f"Sygnał wejściowy: {signal}\n")

    fft_result = fft_cooley_tukey(signal)

    print("Wynik FFT (Liczby zespolone: Część Rzeczywista + Urojona):")
    for i, val in enumerate(fft_result):
        print(f"X[{i}] = {val.real: .3f} + {val.imag: .3f}j")

Sygnał wejściowy: [1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0]

Wynik FFT (Liczby zespolone: Część Rzeczywista + Urojona):
X[0] =  3.000 +  0.000j
X[1] =  0.707 + -0.707j
X[2] =  2.000 + -1.000j
X[3] = -0.707 + -0.707j
X[4] =  1.000 +  0.000j
X[5] = -0.707 +  0.707j
X[6] =  2.000 +  1.000j
X[7] =  0.707 +  0.707j


## 4. Sprzętowa implementacja (SystemVerilog)

Nasz kod implementuje akcelerator Szybkiej Transformaty Fouriera (FFT) oparty na algorytmie Radix-2. Został zaprojektowany w języku SystemVerilog z myślą o syntezie w układach FPGA/ASIC. Główną zaletą tej implementacji jest zastosowanie przetwarzania potokowego (pipeline), co znacznie zwiększa przepustowość układu.

Architektura składa się z następujących modułów:
* **fft_top**: Główny moduł integrujący. Zarządza 5-taktowym rurociągiem opóźniającym, synchronizując adresy zapisu i odczytu z jednostką obliczeniową.
* **control_unit**: Sprzętowy sterownik generujący adresy odczytu i zapisu dla pamięci RAM i ROM. Unika kosztownych operacji dzielenia, wykorzystując zamiast tego szybkie przesunięcia bitowe.
* **twiddle_rom**: Pamięć stała zawierająca prekompilowane wartości trygonometryczne (współczynniki obrotu).
* **dual_port_ram**: Pamięć operacyjna z dwoma niezależnymi portami. Przechowuje wartości wejściowe, wyniki pośrednie etapów (motylków) oraz ostateczny wynik.
* **butterfly_unit**: 3-taktowa jednostka arytmetyczna. Odpowiada za mnożenie przez współczynniki obrotu oraz równoległe dodawanie i odejmowanie (operacja motylkowa) z uwzględnieniem zapobiegania przepełnieniom.